In [1]:
import pyffx
import os

class DIYTokenizer:
    def __init__(self, secret_key: bytes):
        """
        secret_key: กุญแจลับสำหรับเข้ารหัส (สำคัญมาก ห้ามหลุดเด็ดขาด)
        """
        self.secret_key = secret_key
        
        # ตั้งค่าตัวอักษรที่อนุญาต (Alphabet) ในที่นี้คือตัวเลข 0-9
        self.alphabet = '0123456789'

    def tokenize_thai_id(self, citizen_id: str) -> str:
        """แปลงเลขบัตรประชาชน 13 หลัก เป็น Token 13 หลัก"""
        if len(citizen_id) != 13 or not citizen_id.isdigit():
            raise ValueError("บัตรประชาชนต้องเป็นตัวเลข 13 หลัก")
            
        # กำหนดความยาวให้ FPE Engine
        cipher = pyffx.String(self.secret_key, alphabet=self.alphabet, length=13)
        return cipher.encrypt(citizen_id)

    def detokenize_thai_id(self, token: str) -> str:
        """แปลง Token 13 หลัก คืนเป็นเลขบัตรประชาชนจริง"""
        cipher = pyffx.String(self.secret_key, alphabet=self.alphabet, length=13)
        return cipher.decrypt(token)

    def tokenize_credit_card(self, cc_number: str) -> str:
        """แปลงเลขบัตรเครดิต 16 หลัก เป็น Token 16 หลัก (รักษา 6 หลักแรก และ 4 หลักสุดท้าย)"""
        cc_clean = cc_number.replace("-", "").replace(" ", "")
        if len(cc_clean) != 16:
            raise ValueError("เลขบัตรเครดิตต้องมี 16 หลัก")
            
        # เพื่อประโยชน์ในการทำ Routing และแสดงผล มักจะเก็บ 6 หลักแรก (BIN) และ 4 หลักท้ายไว้
        # เราจะ Tokenize แค่ 6 หลักตรงกลาง
        bin_part = cc_clean[:6]
        mid_part = cc_clean[6:12]
        last_part = cc_clean[12:]
        
        cipher = pyffx.String(self.secret_key, alphabet=self.alphabet, length=6)
        token_mid = cipher.encrypt(mid_part)
        
        return f"{bin_part}{token_mid}{last_part}"


# ==========================================
# ทดสอบการใช้งานจริง
# ==========================================
if __name__ == "__main__":
    # ⚠️ ในระบบจริง กุญแจนี้ต้องดึงมาจาก KMS (Key Management Service) หรือ Environment Variable 
    # ห้าม Hardcode ไว้ใน Source code เด็ดขาด
    MASTER_KEY = b"my-super-secret-key-256-bits!!" 
    
    tokenizer = DIYTokenizer(secret_key=MASTER_KEY)
    
    print("--- ทดสอบบัตรประชาชน ---")
    real_id = "1100200300400"
    token_id = tokenizer.tokenize_thai_id(real_id)
    recovered_id = tokenizer.detokenize_thai_id(token_id)
    
    print(f"ข้อมูลจริง: {real_id}")
    print(f"Token:    {token_id}")
    print(f"ถอดรหัส:   {recovered_id}")
    assert real_id == recovered_id
    
    print("\n--- ทดสอบบัตรเครดิต (Masking FPE) ---")
    real_cc = "4111222233334444"
    token_cc = tokenizer.tokenize_credit_card(real_cc)
    
    print(f"ข้อมูลจริง: {real_cc}")
    print(f"Token:    {token_cc} (เก็บ 6 หน้า 4 หลัง)")

--- ทดสอบบัตรประชาชน ---
ข้อมูลจริง: 1100200300400
Token:    8667307847498
ถอดรหัส:   1100200300400

--- ทดสอบบัตรเครดิต (Masking FPE) ---
ข้อมูลจริง: 4111222233334444
Token:    4111221051364444 (เก็บ 6 หน้า 4 หลัง)


In [4]:
from ff3 import FF3Cipher

# ต้องใช้ Key ขนาด 128-bit (16 bytes) หรือ 192-bit หรือ 256-bit เป็น Hex string
key = "2B7E151628AED2A6ABF7158809CF4F3C"
tweak = "D8E7920AFA330A73" # ค่า Tweak ช่วยเพิ่มความปลอดภัยแบบสุ่ม

cipher = FF3Cipher(key, tweak)

# เข้ารหัสบัตรประชาชน 13 หลัก
real_id = "1100200300400"
token = cipher.encrypt(real_id)

print(f"ข้อมูลต้นฉบับ: {real_id}")
print(f"Token (เข้ารหัส): {token}")

# ถอดรหัส
decrypted = cipher.decrypt(token)

# พิมพ์ผลลัพธ์การถอดรหัส
print(f"ข้อมูลถอดรหัส: {decrypted}")

ข้อมูลต้นฉบับ: 1100200300400
Token (เข้ารหัส): 8669010845197
ข้อมูลถอดรหัส: 1100200300400


HMAC (Hash-based Message Authentication Code) one_way_tokenize

In [ ]:
import hashlib
import hmac

def one_way_tokenize(data: str, secret_key: bytes) -> str:
    # ใช้ SHA-256 ร่วมกับ Secret Key (ป้องกันการถูกโจมตีแบบ Rainbow Table)
    hashed = hmac.new(secret_key, data.encode('utf-8'), hashlib.sha256)
    # ตัดมาใช้แค่ 16 ตัวอักษร เพื่อให้ Token ไม่ยาวเกินไป
    return f"tok_{hashed.hexdigest()[:100]}"

secret = b"my-analytics-salt-key"
print(one_way_tokenize("somchai@example.com", secret))

tok_bc492f06ec9e314dceaa0b8314e1b2bfea68be73ab95485bd8d8c784f731a558


ลองใช้กับ log ocsf

Format-Preserving Encryption (FPE): คือการเข้ารหัสที่ "ผลลัพธ์ (Ciphertext) จะต้องมีรูปแบบ ความยาว และชนิดของตัวอักษรเหมือนกับข้อมูลต้นฉบับ (Plaintext) ทุกประการ" 

FF3 / FF3-1 (Feistel-based Format-preserving encryption): เป็นชื่อของอัลกอริทึมเฉพาะที่ใช้ในโค้ด (มาจากไลบรารี ff3) ซึ่งเป็นมาตรฐานที่ได้รับการรับรองจากสถาบัน NIST ของสหรัฐอเมริกา

In [8]:
import json
from ff3 import FF3Cipher

# ==========================================
# 1. ตั้งค่า Key และ Tweak
# ==========================================
key = "2B7E151628AED2A6ABF7158809CF4F3C"
tweak = "D8E7920AFA330A73"

# Cipher สำหรับตัวเลข (Radix=10) ใช้กับ IP Address
cipher_num = FF3Cipher(key, tweak, radix=10)

# Cipher สำหรับตัวอักษรและตัวเลข (Radix=62) ใช้กับ Username
cipher_text = FF3Cipher(key, tweak, radix=62)

# ==========================================
# 2. ฟังก์ชันช่วยเหลือสำหรับ IP Address
# ==========================================
def encrypt_ip(ip_str: str, cipher: FF3Cipher) -> str:
    """ถอดจุดออก -> เข้ารหัสเฉพาะตัวเลข -> ใส่จุดกลับที่เดิม (Format-Preserving)"""
    digits_only = "".join(c for c in ip_str if c.isdigit())
    encrypted_digits = cipher.encrypt(digits_only)
    
    result = []
    idx = 0
    for char in ip_str:
        if char == '.':
            result.append('.')
        else:
            result.append(encrypted_digits[idx])
            idx += 1
    return "".join(result)

def decrypt_ip(ip_str: str, cipher: FF3Cipher) -> str:
    """ทำกระบวนการย้อนกลับสำหรับ IP"""
    digits_only = "".join(c for c in ip_str if c.isdigit())
    decrypted_digits = cipher.decrypt(digits_only)
    
    result = []
    idx = 0
    for char in ip_str:
        if char == '.':
            result.append('.')
        else:
            result.append(decrypted_digits[idx])
            idx += 1
    return "".join(result)

# ==========================================
# 3. ข้อมูล Log ต้นฉบับ
# ==========================================
log_data = {
  "activity_name": "Create",
  "time": 1788369641000,
  "message": "Malware detected and quarantined successfully",
  "device": {
    "hostname": "PC-FINANCE-01",
    "ip": "172.20.28.55",  # เป้าหมายที่ 1
  },
  "evidences": [
    {
      "name": "Malware file evidence",
      "device": {
        "hostname": "PC-FINANCE-01",
        "ip": "172.20.28.55",  # เป้าหมายที่ 2
      },
      "user": {
        "name": "administrator", # เป้าหมายที่ 3
        "type_id": 2
      }
    }
  ]
}

print("--- [1] ข้อมูลก่อนเข้ารหัส ---")
print(f"Device IP : {log_data['device']['ip']}")
print(f"User Name : {log_data['evidences'][0]['user']['name']}")

# ==========================================
# 4. ทำการเข้ารหัส (Tokenization)
# ==========================================
# เข้ารหัส IP
log_data["device"]["ip"] = encrypt_ip(log_data["device"]["ip"], cipher_num)
log_data["evidences"][0]["device"]["ip"] = encrypt_ip(log_data["evidences"][0]["device"]["ip"], cipher_num)

# เข้ารหัส Username
original_user = log_data["evidences"][0]["user"]["name"]
log_data["evidences"][0]["user"]["name"] = cipher_text.encrypt(original_user)

print("\n--- [2] ข้อมูลหลังเข้ารหัส (ลง Database/SIEM แบบนี้) ---")
print(json.dumps(log_data, indent=2))

# ==========================================
# 5. ทำการถอดรหัสคืน (Detokenization)
# ==========================================
recovered_ip = decrypt_ip(log_data["device"]["ip"], cipher_num)
recovered_user = cipher_text.decrypt(log_data["evidences"][0]["user"]["name"])

print("\n--- [3] ทดสอบถอดรหัสคืนข้อมูลจริง ---")
print(f"Recovered Device IP : {recovered_ip}")
print(f"Recovered User Name : {recovered_user}")

--- [1] ข้อมูลก่อนเข้ารหัส ---
Device IP : 172.20.28.55
User Name : administrator

--- [2] ข้อมูลหลังเข้ารหัส (ลง Database/SIEM แบบนี้) ---
{
  "activity_name": "Create",
  "time": 1788369641000,
  "message": "Malware detected and quarantined successfully",
  "device": {
    "hostname": "PC-FINANCE-01",
    "ip": "488.18.06.75"
  },
  "evidences": [
    {
      "name": "Malware file evidence",
      "device": {
        "hostname": "PC-FINANCE-01",
        "ip": "488.18.06.75"
      },
      "user": {
        "name": "ilft6SZrfAMor",
        "type_id": 2
      }
    }
  ]
}

--- [3] ทดสอบถอดรหัสคืนข้อมูลจริง ---
Recovered Device IP : 172.20.28.55
Recovered User Name : administrator


In [ ]:
import json
import re
from ff3 import FF3Cipher

# ==========================================
# ทำ FPE (FF3/FF3-1) กับข้อมูลจริงจากไฟล์ OCSF log (ocsf.log)
# เป้าหมาย: Tokenize field ที่อ่อนไหว เช่น hostname, ip, username, agent uid
# โดยยังคงรูปแบบ [XXX_NN] เดิมไว้ (Format-Preserving)
# ==========================================

# 128-bit key (32 hex chars) และ tweak (8 byte / 16 hex chars) สำหรับ FF3-1
# ⚠️ ในระบบจริงต้องดึงจาก KMS/Environment Variable ห้าม Hardcode
FF3_KEY = "F3D2FE0E66B2AA56806944AA2770D53A"
FF3_TWEAK = "6A81B1DC1D04B6CA"

# Field ในไฟล์นี้เป็น placeholder รูปแบบ [ตัวอักษรพิมพ์ใหญ่/ตัวเลข/underscore]
# เช่น [HOST_01], [INT_IP_01], [USER_01], [AGENT_01]
# จึงต้องสร้าง Alphabet เอง (FF3-1 รองรับ custom alphabet ต่างจาก FF3 เดิม)
ALPHABET = "ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_"
field_cipher = FF3Cipher.withCustomAlphabet(FF3_KEY, FF3_TWEAK, ALPHABET)

PLACEHOLDER_RE = re.compile(r"\[[A-Za-z0-9_]+\]")


def tokenize_field(cipher: FF3Cipher, value: str) -> str:
    """Tokenize ข้อความภายในวงเล็บ [] แล้วคืนค่าในรูปแบบ [] เดิม"""
    inner = value[1:-1]
    return f"[{cipher.encrypt(inner)}]"


def walk_and_tokenize(node, cipher: FF3Cipher):
    """เดิน JSON แบบ recursive แล้ว Tokenize ทุก field ที่อยู่ในรูปแบบ [XXX_NN]"""
    if isinstance(node, dict):
        return {key: walk_and_tokenize(val, cipher) for key, val in node.items()}
    if isinstance(node, list):
        return [walk_and_tokenize(val, cipher) for val in node]
    if isinstance(node, str) and PLACEHOLDER_RE.fullmatch(node):
        return tokenize_field(cipher, node)
    return node


with open("ocsf.log", "r", encoding="utf-8") as f:
    ocsf_log = json.load(f)

tokenized_log = walk_and_tokenize(ocsf_log, field_cipher)

print("--- ก่อน Tokenize ---")
print(f"device.hostname        : {ocsf_log['device']['hostname']}")
print(f"device.ip              : {ocsf_log['device']['ip']}")
print(f"device.agent_list[0].uid: {ocsf_log['device']['agent_list'][0]['uid']}")
print(f"evidences[0].user.name : {ocsf_log['evidences'][0]['user']['name']}")

print("\n--- หลัง Tokenize (FF3-1, รูปแบบ [XXX] เดิม) ---")
print(f"device.hostname        : {tokenized_log['device']['hostname']}")
print(f"device.ip              : {tokenized_log['device']['ip']}")
print(f"device.agent_list[0].uid: {tokenized_log['device']['agent_list'][0]['uid']}")
print(f"evidences[0].user.name : {tokenized_log['evidences'][0]['user']['name']}")

# ตรวจสอบว่าสามารถถอดรหัสกลับเป็นค่าเดิมได้ (Reversible)
recovered_hostname = f"[{field_cipher.decrypt(tokenized_log['device']['hostname'][1:-1])}]"
assert recovered_hostname == ocsf_log['device']['hostname']
print(f"\nถอดรหัส device.hostname กลับ: {recovered_hostname}")


--- ก่อน Tokenize ---
device.hostname        : [HOST_01]
device.ip              : [INT_IP_01]
device.agent_list[0].uid: [AGENT_01]
evidences[0].user.name : [USER_01]

--- หลัง Tokenize (FF3-1, รูปแบบ [XXX] เดิม) ---
device.hostname        : [5SDAH7D]
device.ip              : [4LN51D49F]
device.agent_list[0].uid: [858NYC0W]
evidences[0].user.name : [9K_MUDC]

ถอดรหัส device.hostname กลับ: [HOST_01]


: 